# 38 · Scale maths: 10x, 100x and HA/DR
Level L6 · Part 6 · deep · ~20 min · builds on 27, 37

Time: about 20 minutes. This lesson is conceptual maths with small code. No server, no model, no Docker and no keys: every number is either from the course data, a cited source, or a stated assumption. Numbers that are not measured are labelled **DERIVED** or **ASSUMED**.

## Goal
By the end you can extend a capacity model from 1x to 10x and 100x traffic, show where the zone and region rules add GPUs, and name the RPO (data you can lose) and the RTO (time until the service serves again) for each failure domain.

## Why it matters
- **Without it:** you size the fleet for today's peak and call it highly available. At 10x the fleet is about 125 GPUs spread over three zones. Lose one zone and the other two run at about 104% of what the peak needs: the queue grows without bound, users time out and retry, and the retries add load. Nothing in the first sizing showed that.
- **With it:** one table gives the GPUs at each traffic level, the extra GPUs the zone and region rules add, and the RPO and RTO each failure domain allows. You can defend every number in a design review, and you can say which number is measured and which is assumed.
- **When you don't need it:** an internal tool with a few dozen users, where a restart of twenty minutes is acceptable and nobody loses money when it is down.

## The idea
Scale is multiplication. Traffic in requests per second becomes output tokens per second, output tokens per second becomes GPUs (at a measured rate per GPU), and GPUs become cost per day. Every redundancy rule then multiplies the fleet again: a spare for each zone, or a full copy for each region.

Two words carry the failure side. **RPO** (recovery point objective) is the work or data you can lose, counted back in time. **RTO** (recovery time objective) is how long the service is down before it serves again. They are separate: a service can lose nothing (RPO zero) and still be down for ten minutes (RTO ten minutes).

Think of a restaurant with two kitchens. If one kitchen closes mid-service, the RPO is the orders already taken but not cooked. The RTO is how long until guests are served again: seconds if the other kitchen has room, hours if it must hire and train.

## Picture
```mermaid
flowchart TD
  G[Global routing] --> RA[Region A, sized for 100% of peak]
  G --> RB[Region B, sized for 100% of peak]
  RA --> Z1[Zone 1: replicas]
  RA --> Z2[Zone 2: replicas]
  RA --> Z3[Zone 3: replicas]
  Z1 --> E[Engine replica: GPUs, KV cache]
  RA --> GW[Gateway and budget database]
```

## Already know this?
This is the first lesson of Part 6 (Enterprise and the capstone). Lessons 38 to 40 use the capacity and failure ideas below. If you can answer all five, skip to lesson 39; lesson 40 is the capstone that uses this maths.

1. What is the difference between RPO and RTO?
2. A region carrying half your peak goes down. Can the other region carry the peak, and what decides it?
3. Why does a replica's KV cache not survive that replica's crash by default?
4. Why is goodput (lesson 27), not raw tokens per second, the number to size with?
5. Why is one vLLM server shared by three tenants not an isolation boundary (lesson 36)?

Answers: 1 is above; 2 is Step 4; 3 is the engine-replica row of the Step 6 table below; 4 is lesson 27; 5 is lesson 36.

## Explained
### Step 1: the inputs, and where each one comes from
The one published throughput figure: MLPerf Inference v5.0 reports 33,072 output tokens per second for Llama 2 70B (Server scenario) on 8 H200 GPUs. It is a TensorRT-LLM result, not vLLM. Divide by 8 for the rate per GPU, then by 1.43 for an H100. The 1.43 is the memory-bandwidth ratio (4.8 and 3.35 TB/s, NVIDIA's spec table), not a measured speed. So the H100 rate is **DERIVED**, and an optimistic ceiling.

The 8-GPU figure is an average over that run. It already includes the overhead of splitting the model across the GPUs (tensor parallelism), which we assume here, not measure.

The **measured** rate is yours: lesson 27 measures goodput at your latency target. The course machine has no GPU, so its lesson 27 numbers show the method only.

In [1]:
import json, math
from pathlib import Path

# SOURCED: MLPerf Inference v5.0, Llama 2 70B, Server scenario, 8 x H200 = 33,072 output tokens/s (TensorRT-LLM, not vLLM)
MLPERF_8X_H200_TOK_S = 33_072
# DERIVED: H200 about 1.43x H100 (memory bandwidth 4.8 vs 3.35 TB/s, NVIDIA spec table; a search summary reports MLPerf v4.0 gains of 43-45%, not opened)
H200_OVER_H100 = 1.43
PER_GPU_TOK_S = MLPERF_8X_H200_TOK_S / 8 / H200_OVER_H100   # DERIVED: about 2,891 output tokens/s per H100
MEASURED_TOK_S = None     # YOUR number: output tokens/s per GPU at your latency target (lesson 27, on a GPU)
TOK_S = MEASURED_TOK_S or PER_GPU_TOK_S
TP, TARGET_UTIL, PRICE_GPU_H = 2, 0.7, 2.0     # GPUs per replica; 70% headroom (lesson 26); ASSUMED price per GPU-hour
IN_TOK, OUT_TOK = 1_000, 250                   # planning request size (Brightside; see Step 2 for the trace)
print(f"tokens/s per GPU: {TOK_S:,.0f} ({'measured' if MEASURED_TOK_S else 'DERIVED, not measured'})")

rec = json.loads(Path("../data/recorded/27.json").read_text())["results"]["fixed_0.10"]
cpu_tok_s = rec["goodput_rps"] * rec["mean_out"]        # requests per second that meet the SLO x answer tokens
print(f"lesson 27, 4-core CPU lab: {rec['goodput_rps']} req/s x {rec['mean_out']} tokens = {cpu_tok_s:.1f} tokens/s")
print("a CPU run shows the method, not a GPU rate: put your own GPU number in MEASURED_TOK_S")

tokens/s per GPU: 2,891 (DERIVED, not measured)
lesson 27, 4-core CPU lab: 0.084 req/s x 48 tokens = 4.0 tokens/s
a CPU run shows the method, not a GPU rate: put your own GPU number in MEASURED_TOK_S


### Step 2: from traffic to GPUs, at 1x, 10x and 100x
Three steps per traffic level. Decode tokens per second = requests per second × answer tokens. GPUs at the ceiling = that rate ÷ the rate per GPU. GPUs with headroom = the ceiling ÷ 70%, so bursts and one lost GPU have room. Replicas are two GPUs each. The prefill work (1,000 input tokens per request, 10 million per second at 100x) is not in these GPU counts: it needs its own pool or share of the GPUs, and lesson 27 measures it.

In [2]:
def plan(qps, out_tok=OUT_TOK):
    """GPUs for decode at the rate per GPU, then with headroom, replicas and cost per day."""
    ceiling = math.ceil(qps * out_tok / TOK_S)          # every GPU running at the rate above
    planned = math.ceil(ceiling / TARGET_UTIL)          # run at 70%, so bursts and a loss have room
    return {"ceiling": ceiling, "planned": planned,
            "replicas": math.ceil(planned / TP), "cost_day": planned * PRICE_GPU_H * 24}

print(f"{'traffic':>8} {'decode tok/s':>13} {'GPUs at ceiling':>16} {'GPUs at 70%':>12} {'TP=2 replicas':>14} {'cost/day':>10}")
for label, qps in (("1x", 100), ("10x", 1_000), ("100x", 10_000)):
    p = plan(qps)
    print(f"{label:>8} {qps * OUT_TOK:>13,} {p['ceiling']:>16,} {p['planned']:>12,} {p['replicas']:>14,} ${p['cost_day']:>9,.0f}")

 traffic  decode tok/s  GPUs at ceiling  GPUs at 70%  TP=2 replicas   cost/day
      1x        25,000                9           13              7 $      624
     10x       250,000               87          125             63 $    6,000
    100x     2,500,000              865        1,236            618 $   59,328


**What you should see:** the ceiling is the number a vendor would quote, and the planned column is the one you buy. Every GPU count here is for decode only: the prefill work is not in it (Step 2). The cost column uses the assumed $2 per GPU-hour, so it shows the scale of the bill, not a price. The 100x row is the capstone target of lesson 40, before any zone or region rule.

### Step 3: check the slots too (Little's law, lesson 26)
Tokens per second is not the only limit. Each request holds one slot for its prefill plus its answer, so the requests in flight are L = λ × W (lesson 26). A replica can hold only as many requests as its KV cache allows. The KV numbers come from the Llama 3.1 70B shape (80 layers, 8 KV heads, head size 128) at FP8. The 5 GB of activations per GPU is an **ASSUMPTION**. Check both limits and keep the larger replica count.

In [3]:
PREFILL_S, TPOT_S = 0.25, 0.025                          # lesson 26 assumptions, per request (ASSUMED; measure in lesson 27)
W_S = PREFILL_S + OUT_TOK * TPOT_S                        # seconds one request holds a slot
KV_BYTES_PER_TOKEN = 2 * 80 * 8 * 128 * 1                 # K and V x 80 layers x 8 KV heads x 128 x 1 byte (FP8): 160 KiB
KV_BYTES_REPLICA = (80 * 0.92 - 35 - 5) * 2 * 1e9         # per GPU: 80 GB x 0.92 (gpu_memory_utilization, lesson 16), less 35 GB weights (70 GB / 2 GPUs) and 5 GB activations (ASSUMED); x 2 GPUs
SLOTS = int(KV_BYTES_REPLICA // KV_BYTES_PER_TOKEN // (IN_TOK + OUT_TOK))   # requests one replica holds at once
print(f"W = {W_S:.1f} s per request; KV per replica {KV_BYTES_REPLICA / 1e9:.1f} GB; {SLOTS} requests per replica")
print(f"{'traffic':>8} {'in flight':>10} {'replicas for slots':>19} {'replicas for tokens/s':>22}")
for label, qps in (("1x", 100), ("10x", 1_000), ("100x", 10_000)):
    in_flight = qps * W_S
    by_slots = math.ceil(in_flight / (SLOTS * TARGET_UTIL))
    print(f"{label:>8} {in_flight:>10,.0f} {by_slots:>19,} {plan(qps)['replicas']:>22,}")

W = 6.5 s per request; KV per replica 67.2 GB; 328 requests per replica
 traffic  in flight  replicas for slots  replicas for tokens/s
      1x        650                   3                      7
     10x      6,500                  29                     63
    100x     65,000                 284                    618


**What you should see:** at every level the tokens-per-second count is the larger one, so throughput binds, not the cache. At 100x the requests in flight are 65,000. More cache would not help; more GPUs or shorter answers would. Keep both checks in the design, because a longer answer flips the result.

### Step 4: failure domains add GPUs
A plan that holds the peak only when every GPU is up is not a plan. The test is simple: remove a failure domain's share of GPUs, and see whether the survivors stay below 100%. `util_after_loss` returns that share. Above 1.0 (100%) the queue grows without bound, which is an outage. Three zones are assumed.

In [4]:
def util_after_loss(qps, gpus, lost, out_tok=OUT_TOK):
    """Share of the survivors' token rate that the peak needs: 1.0 is full."""
    return qps * out_tok / ((gpus - lost) * TOK_S)

def must_hold(util):
    """Raise if the survivors cannot hold the peak."""
    if util >= 1:
        raise ValueError(f"survivors at {util:.0%}: the queue grows without bound")
    return util

ZONES = 3
p = plan(1_000)                                              # 10x
lost_zone = math.ceil(p["planned"] / ZONES)                  # one zone of three goes down
print(f"10x, {p['planned']} GPUs over {ZONES} zones: losing a zone ({lost_zone} GPUs) leaves survivors at "
      f"{util_after_loss(1_000, p['planned'], lost_zone):.0%}")
zone_proof = math.ceil(p["planned"] * ZONES / (ZONES - 1))   # the two surviving zones must still run at 70%
lost_zone2 = math.ceil(zone_proof / ZONES)
print(f"zone-proof plan: {zone_proof} GPUs; losing a zone leaves {util_after_loss(1_000, zone_proof, lost_zone2):.0%}")
print(f"two regions, each zone-proof and each carrying the full peak: {2 * zone_proof} GPUs, "
      f"about ${2 * zone_proof * PRICE_GPU_H * 24:,.0f} a day at the assumed price")

10x, 125 GPUs over 3 zones: losing a zone (42 GPUs) leaves survivors at 104%
zone-proof plan: 188 GPUs; losing a zone leaves 69%
two regions, each zone-proof and each carrying the full peak: 376 GPUs, about $18,048 a day at the assumed price


**What you should see:** every GPU count in this step is for decode only (prefill is excluded, Step 2). The planned fleet of 125 GPUs falls to 104% of the peak when a zone goes. The zone-proof fleet of 188 GPUs holds at about 69%. A region rule doubles the bill again. Two regions with zone-proof fleets is the design a capstone must justify, or else a degraded mode must be written down.

### Step 5: how long until a replica serves again (RTO inputs)
Two RTO inputs are measured or sourced here. The first is the **weight load**: a replica that is lost must read its weights before it can serve, so the time is weight bytes divided by the read rate. The read rates are **ASSUMED**; the lab cannot measure them. The second is lesson 37's recorded run: three simulated replicas, one killed under load, and the gateway's retries.

In [5]:
WEIGHTS_GB = 70                          # Llama 3.1 70B at FP8 is about 1 byte per parameter (ASSUMED; check your checkpoint size)
for rate in (2.0, 0.5, 0.2):            # ASSUMED sustained read rates for one replica, GB/s
    print(f"read at {rate} GB/s: {WEIGHTS_GB / rate:,.0f} s to load the weights (engine start and warm-up come on top)")
rec37 = json.loads(Path("../data/recorded/37.json").read_text())["phases"]["node loss"]
print(f"lesson 37 (recorded, SIMULATED replicas): {rec37['ok']} of 120 requests answered; "
      f"{rec37['503']} refused with 503 and {rec37['conn_error']} connection errors were retried; failed after retries: {rec37['failed']}")

read at 2.0 GB/s: 35 s to load the weights (engine start and warm-up come on top)
read at 0.5 GB/s: 140 s to load the weights (engine start and warm-up come on top)
read at 0.2 GB/s: 350 s to load the weights (engine start and warm-up come on top)
lesson 37 (recorded, SIMULATED replicas): 120 of 120 requests answered; 3 refused with 503 and 5 connection errors were retried; failed after retries: 0


**What you should see:** a lost replica takes 35 to 350 seconds of weight loading before it adds capacity again, before engine start-up, which is why the survivors must already hold the load. The lab run shows the traffic side: the retries moved every request to a survivor, so no request failed. The lab does not show streams cut in the middle, which the table below covers.

### Step 6: RPO and RTO for each failure domain
RPO is data lost; RTO is time until service returns. DERIVED values come from Steps 4 and 5. Sourced values are from the page named, opened 2026-10-23. None is a vendor SLA.

| Failure domain | RPO (data lost) | RTO (serves again) |
|---|---|---|
| Engine replica or GPU | 0 stored; in-flight streams are cut and the client retries. KV cache lives in GPU memory and is lost with the replica; the replacement starts cold. | Seconds for traffic (lesson 37: 120 of 120 answered); capacity 35 to 350 s DERIVED weight load |
| Node (TP group) | 0 | Restart plus weight load, 35 to 350 s DERIVED; survivors must hold the peak meanwhile |
| Zone | 0 | Seconds with a zone-proof fleet (69%); otherwise minutes of queueing and timeouts |
| Region | 0 for replicated weights; gateway data at its database lag | Your global failover time (not measured here); capacity only if the other region holds full peak |
| Gateway database (budgets, keys) | Replication lag (Postgres, not quantified here) | Serving continues if `allow_requests_on_db_unavailable` is set (private VPC only, LiteLLM docs); budgets fail open |
| Redis (rate-limit counters) | Counters since last persistence | Fail open or fail closed (503), by setting (LiteLLM docs) |
| Weights bucket | Dual-region default: up to 12 h (100% within 12 h; 99.9% within 1 h). Turbo: 15 min, premium | Designed as zero for dual- and multi-region storage (Google Cloud docs); a regional bucket is down with its region |

Engine and node failures cost time, not data. A region failure costs time only if the second region already holds the peak.

## Worked example
A 10x launch: 1,000 requests per second, 250 answer tokens each, one model, three zones per region, two regions.

1. Output tokens per second: 1,000 × 250 = 250,000.
2. GPUs at the ceiling (decode only): 250,000 ÷ 2,891 (DERIVED rate per H100) = 87.
3. GPUs at 70%: 87 ÷ 0.7 = 125. A zone loss takes 42 of them and leaves 104%: an outage.
4. Zone-proof per region: 125 × 3/2 = 188 GPUs. A zone loss leaves about 69% (decode only).
5. Two regions, each carrying full peak: 2 × 188 = 376 GPUs, about $18,000 a day at the assumed $2 per GPU-hour.

The launch passes the test only at step 4. The weights are the other half: the second region must hold the new model version before traffic moves to it (Step 6, weights row).

## At real scale
- **DeepSeek, one day in February 2025** (open-infra-index): on average 226.75 nodes of 8 GPUs ran, peak 278; decode about 1.85k output tokens/s per H800. A different model (mixture of experts, wide expert parallelism). Our DERIVED 2,891 per H100 is about 1.6 times higher: the derived ceiling is optimistic, so your lesson 27 number replaces it. → lesson 40. Source: github.com/deepseek-ai/open-infra-index.
- **llm-d wide expert parallelism**: about 3.1k tokens/s per GPU on 16 × 16 B200 (project README, opened 2026-10-23): another setup, so read its settings before you compare. → lesson 39.
- **Failover at the gateway**: open-source nginx marks a server down for 10 seconds after one failure by default. Its `slow_start` ramp is commercial, so a recovered replica with a cold cache takes full traffic at once. → lesson 37 (the 3-node cluster with failover).
- **Go deeper:** Kubernetes topology spread and PodDisruptionBudget keep replicas across zones. Kubernetes objects, pointers only here.

## Decision guide
| Situation | Approach | Why |
|---|---|---|
| Zone loss must not cause queueing | Zone-proof plan: planned × zones ÷ (zones − 1) GPUs per region | Survivors stay at about 70% (69% in Step 4); at 3 zones the cost is +50% |
| Region loss must not cause an outage | Two regions, each zone-proof and each carrying full peak | Failover needs capacity; doubling is the price |
| Region loss is acceptable for hours | One region plus a written degraded mode (shorter answers, smaller model) | Cheaper, but the RTO is a decision, not a hope |
| A new model version rolls out | Publish to both buckets, check the replica, then shift traffic | The default RPO is up to 12 h |
| Budgets must hold during a database outage | A replicated database; fail open only inside a private VPC | Without it, spend is not counted while the database is down |
| Load grows from 1x to 10x | Add GPUs or cut answer length first; check the slot count too | At these inputs throughput binds |

## Common misconceptions
**Misconception 1: "the fleet covers the peak, so one lost GPU is fine."** A fleet at 100% of its ceiling has no room. The check below raises the error a queue would produce: the survivors need more than they have.

In [6]:
must_hold(util_after_loss(1_000, 87, 1))      # 87 GPUs with no headroom, one lost: the survivors need 101%

ValueError: survivors at 101%: the queue grows without bound

The fix is the planned count (125) and, for a zone, the zone-proof count (188). **Misconception 2: "RPO zero means no outage."** RPO is about data. A service with RPO zero can still be down for minutes while weights load. **Misconception 3: "two regions cost the same as one."** Each region must carry the full peak unless a degraded mode is written down, so the fleet doubles. **Misconception 4: "the storage RTO is my service's RTO."** The "RTO zero" in the bucket's docs covers the bucket. Your gateway, your database and your engines have their own numbers.

## Try it
The file `../data/07/support_trace.jsonl` is the lesson 07 trace (synthetic). Each line is one request; the field `output_tokens` is its answer length. Its answers are shorter than the planning assumption: on average 69 output tokens, not 250.

Use `plan()` with the trace's mean `output_tokens` as `out_tok` for the 10x traffic (1,000 requests per second). Then work out the zone-proof count with the formula from Step 4: planned × 3 ÷ 2, rounded up. Print the planned GPU count and the zone-proof count.

In [7]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [8]:
trace = [json.loads(l) for l in Path("../data/07/support_trace.jsonl").read_text().splitlines() if l.strip()]
out_trace = sum(r["output_tokens"] for r in trace) / len(trace)
p_trace = plan(1_000, out_tok=out_trace)
zone_trace = math.ceil(p_trace["planned"] * ZONES / (ZONES - 1))
print(f"trace mean {out_trace:.0f} output tokens: {p_trace['planned']} GPUs at 70%, zone-proof {zone_trace}")

trace mean 69 output tokens: 35 GPUs at 70%, zone-proof 53


## Check yourself
1. Region A is lost, and then one zone of region B goes down too. Region B is zone-proof at 10x (188 GPUs). What is its utilisation? *(Answer hidden below.)*
2. The bucket's default replication is 100% within 12 hours. You publish version 4 to region B and fail over three hours later. What is the risk, and what check comes first?
3. The gateway database is down and your pods still answer. What two things stay wrong until it returns?
4. Lesson 37's node-loss run had no failed requests. Why does that not prove the RPO for a cut stream?
5. At 100x the slot check gives 284 replicas and the tokens-per-second check gives 618. Which binds, and what do you change first?

**Answers (hidden):** 1 about 69%: 125 of the 188 GPUs carry the full 250,000 tokens/s. 2 region B may not have version 4 yet; check the replica exists there before you move traffic, or use turbo replication (15 min). 3 budgets are not enforced (they fail open), and the spend of the outage is not counted until the database returns. 4 the lab's retries happened before any answer streamed, and its replicas are simulated, so cut streams are not shown. 5 throughput binds: add GPUs, shorten answers or quantise (lessons 20, 21); more cache does not help.

## Recap
- **RPO is data lost; RTO is time to serve.** They are separate numbers for each failure domain.
- **Traffic becomes GPUs in three steps:** tokens per second, the rate per GPU (measured by you), and headroom at 70%.
- **Check the slots as well as the tokens per second,** and keep the larger replica count (throughput binds at 1x, 10x and 100x here).
- **A zone needs ×1.5 at three zones; a region needs a second full fleet.** Test each failure by removing its share and reading the survivors' utilisation.
- **The 2,891 tokens/s per GPU is DERIVED and optimistic.** Replace it with your own lesson 27 measurement on a GPU before you quote any number.